In [ ]:
!pip install rasterio geopandas ultralytics piexif

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 51.5 MB/s eta 0:00:00


In [ ]:
import os, random, shutil, rasterio, torch, cv2, time, gc, json

import piexif
import numpy as np
import pandas as pd
import geopandas as gpd
from matplotlib import colors, patches
import matplotlib.pyplot as plt

import PIL.Image
import PIL.ExifTags
from tqdm import tqdm
from pathlib import Path

from rasterio.windows import Window
from rasterio.features import shapes
from rasterio.transform import Affine
from shapely.geometry import Polygon, Point, MultiPolygon, box, shape

from ultralytics import YOLO
from IPython.display import Image

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Memory efficient
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('medium')

# Load model
model = YOLO('/content/drive/MyDrive/BlueCARES_Seagrass/field_processing/Classification/yolov11l-benthic-classification-100epochs-v01/weights/best.pt')
model.fuse()  # Fuse layers for inference speed
model.to('cuda' if torch.cuda.is_available() else 'cpu')

def get_gps_from_exif(exif_data):
    """Extract GPS coordinates from EXIF data"""
    if not exif_data:
        return None

    gps_info = None

    # Direct GPS tag
    if 34853 in exif_data:
        gps_info = exif_data[34853]

    # GPSInfo tag name
    for tag_id, value in exif_data.items():
        tag_name = PIL.ExifTags.TAGS.get(tag_id, tag_id)
        if 'gps' in str(tag_name).lower():
            gps_info = value
            break

    if not gps_info:
        return None

    # Parse GPS coordinates
    def convert_to_degrees(value):
        """Convert GPS coordinates stored in EXIF to degrees in float"""
        d = float(value[0])
        m = float(value[1])
        s = float(value[2])
        return d + (m / 60.0) + (s / 3600.0)

    gps_data = {}

    # GPS Latitude
    if 2 in gps_info and 1 in gps_info:
        lat = convert_to_degrees(gps_info[2])
        if gps_info[1] == 'S':  # South is negative
            lat = -lat
        gps_data['latitude'] = lat

    # GPS Longitude
    if 4 in gps_info and 3 in gps_info:
        lon = convert_to_degrees(gps_info[4])
        if gps_info[3] == 'W':  # West is negative
            lon = -lon
        gps_data['longitude'] = lon

    # GPS Altitude
    if 6 in gps_info:
        gps_data['altitude'] = float(gps_info[6])

    # GPS Timestamp
    if 7 in gps_info:
        gps_data['timestamp'] = str(gps_info[7])

    # GPS Date
    if 29 in gps_info:
        gps_data['date'] = str(gps_info[29])

    return gps_data


def copy_exif_to_image(source_path, target_path):
    """Copy EXIF metadata from source image to target image"""
    try:
        # Load EXIF data from source
        exif_dict = piexif.load(str(source_path))

        # Load target image using explicit PIL.Image
        target_img = PIL.Image.open(target_path)

        # Convert EXIF to bytes
        exif_bytes = piexif.dump(exif_dict)

        # Save target image with EXIF data
        target_img.save(target_path, exif=exif_bytes, quality=95)
        target_img.close()

        return True

    except Exception as e:
        print(f"  Warning: Could not copy EXIF data: {e}")
        return False


def classify_benthic(input_dir, output_dir, img_size=640, conf=0.20, save=True):
    """Classify benthic images using YOLO and retain image metadata including GPS"""

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    os.makedirs(output_dir, exist_ok=True)

    input_dir = Path(input_dir)
    images = sorted([
        str(p) for p in input_dir.iterdir() if p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.tiff', '.tif']
    ])

    results_summary = []

    start_time = time.time()

    with torch.inference_mode():
        for image_path in images:
            print(f"Processing: {Path(image_path).name}")

            # Read image metadata before processing - using explicit PIL.Image
            img = PIL.Image.open(image_path)

            # Get basic image metadata
            img_width, img_height = img.size
            img_mode = img.mode
            img_format = img.format

            # Get EXIF/metadata
            exif_data = {}
            gps_coordinates = None

            try:
                if hasattr(img, '_getexif') and img._getexif():
                    exif = img._getexif()
                    if exif:
                        # Convert all EXIF tags to readable format
                        for tag_id, value in exif.items():
                            tag_name = PIL.ExifTags.TAGS.get(tag_id, tag_id)
                            # Convert bytes to string if needed
                            if isinstance(value, bytes):
                                try:
                                    value = value.decode('utf-8', errors='ignore')
                                except:
                                    value = str(value)
                            exif_data[tag_name] = value

                        # Extract GPS coordinates
                        gps_coordinates = get_gps_from_exif(exif)
            except Exception as e:
                print(f"  Warning: Could not read EXIF data: {e}")
                exif_data = {}

            # Also try using piexif for more comprehensive EXIF reading
            try:
                exif_dict = piexif.load(str(image_path))
                if 'GPS' in exif_dict and exif_dict['GPS']:
                    gps_data = exif_dict['GPS']
                    # Parse GPS data using piexif
                    gps_coordinates = {}

                    # GPS Latitude
                    if piexif.GPSIFD.GPSLatitude in gps_data:
                        lat_data = gps_data[piexif.GPSIFD.GPSLatitude]
                        lat_ref = gps_data.get(piexif.GPSIFD.GPSLatitudeRef, b'N')
                        if lat_data:
                            lat = lat_data[0][0] / lat_data[0][1] + \
                                  lat_data[1][0] / lat_data[1][1] / 60 + \
                                  lat_data[2][0] / lat_data[2][1] / 3600
                            if lat_ref == b'S':
                                lat = -lat
                            gps_coordinates['latitude'] = lat

                    # GPS Longitude
                    if piexif.GPSIFD.GPSLongitude in gps_data:
                        lon_data = gps_data[piexif.GPSIFD.GPSLongitude]
                        lon_ref = gps_data.get(piexif.GPSIFD.GPSLongitudeRef, b'E')
                        if lon_data:
                            lon = lon_data[0][0] / lon_data[0][1] + \
                                  lon_data[1][0] / lon_data[1][1] / 60 + \
                                  lon_data[2][0] / lon_data[2][1] / 3600
                            if lon_ref == b'W':
                                lon = -lon
                            gps_coordinates['longitude'] = lon

                    # GPS Altitude
                    if piexif.GPSIFD.GPSAltitude in gps_data:
                        alt_data = gps_data[piexif.GPSIFD.GPSAltitude]
                        if alt_data:
                            gps_coordinates['altitude'] = alt_data[0] / alt_data[1]

                    # GPS Timestamp
                    if piexif.GPSIFD.GPSTimeStamp in gps_data:
                        gps_coordinates['timestamp'] = str(gps_data[piexif.GPSIFD.GPSTimeStamp])

            except Exception as e:
                print(f"  Note: Could not read GPS with piexif: {e}")

            img.close()

            # Process the image for classification
            result = model(
                image_path,
                imgsz=img_size,
                conf=conf,
                device=device,
                save=save,
                project=output_dir,
                name="predictions",
                exist_ok=True,
                verbose=False
            )[0]

            # Get classification results
            classification_info = {}
            if hasattr(result, 'probs') and result.probs is not None:
                probs = result.probs
                # Get top classification
                top1_conf, top1_idx = torch.max(probs.data, dim=0)
                top1_class = result.names[int(top1_idx)]

                # Get all classes with confidences
                all_classes = {}
                for i, class_name in result.names.items():
                    if i < len(probs.data):
                        all_classes[class_name] = float(probs.data[i])

                classification_info = {
                    'predicted_class': top1_class,
                    'confidence': float(top1_conf),
                    'class_id': int(top1_idx),
                    'all_classes': all_classes,
                    'top5_classes': dict(sorted(all_classes.items(),
                                                key=lambda x: x[1],
                                                reverse=True)[:5])
                }

            # ========== Copy EXIF metadata to output image ==========
            output_image_path = None
            predictions_dir = Path(output_dir) / "predictions"

            if predictions_dir.exists():
                # Look for the output image with the same name
                possible_output = predictions_dir / Path(image_path).name
                if possible_output.exists():
                    output_image_path = possible_output
                else:
                    for ext in ['.jpg', '.jpeg', '.png']:
                        possible_output = predictions_dir / (Path(image_path).stem + ext)
                        if possible_output.exists():
                            output_image_path = possible_output
                            break

            # Copy EXIF data to the output image
            metadata_copied = False
            if output_image_path and output_image_path.exists():
                metadata_copied = copy_exif_to_image(image_path, output_image_path)
                if metadata_copied:
                    print(f"  ✓ Metadata copied to output image")
                else:
                    print(f"  ✗ Could not copy metadata to output image")

            if output_image_path and output_image_path.exists():
                if classification_info and 'predicted_class' in classification_info:
                    top1_class = classification_info['predicted_class']

                    new_name = f"{top1_class}{output_image_path.suffix}"
                    new_path = output_image_path.parent / new_name

                    counter = 1
                    while new_path.exists():
                        new_name = f"{top1_class}_{counter}{output_image_path.suffix}"
                        new_path = output_image_path.parent / new_name
                        counter += 1

                    output_image_path.rename(new_path)
                    output_image_path = new_path
            # ==============================================================

            # Store results with GPS and metadata
            image_result = {
                "filename": Path(image_path).name,
                "filepath": str(image_path),
                "output_filepath": str(output_image_path) if output_image_path else None,
                "metadata_preserved": metadata_copied,
                "original_dimensions": {
                    "width_pixels": img_width,
                    "height_pixels": img_height
                },
                "gps_coordinates": gps_coordinates,
                "image_metadata": {
                    "format": img_format,
                    "color_mode": img_mode,
                    "has_exif": len(exif_data) > 0,
                    "exif_keys": list(exif_data.keys()) if exif_data else []
                },
                "classification": classification_info,
                "processing_info": {
                    "model_input_size": img_size,
                    "confidence_threshold": conf,
                    "processing_time": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "device_used": device
                }
            }

            results_summary.append(image_result)

            # Clear memory
            del result
            torch.cuda.empty_cache()

    elapsed = time.time() - start_time
    print(f"\n✅ Processed {len(images)} images in {elapsed:.2f}s ({elapsed/len(images):.2f}s per image)")

    # Save detailed results to JSON
    results_json_path = Path(output_dir) / "classification_results_with_gps.json"
    with open(results_json_path, 'w') as f:
        json.dump(results_summary, f, indent=2, default=str)

    csv_data = []
    for result in results_summary:
        csv_row = {
            'filename': result['filename'],
            'predicted_class': result['classification'].get('predicted_class', 'N/A'),
            'confidence': result['classification'].get('confidence', 0),
            'latitude': result['gps_coordinates'].get('latitude', 'N/A') if result['gps_coordinates'] else 'N/A',
            'longitude': result['gps_coordinates'].get('longitude', 'N/A') if result['gps_coordinates'] else 'N/A',
            'width': result['original_dimensions']['width_pixels'],
            'height': result['original_dimensions']['height_pixels'],
            'metadata_preserved': result['metadata_preserved']
        }
        csv_data.append(csv_row)

    df = pd.DataFrame(csv_data)
    csv_path = Path(output_dir) / "summary.csv"
    df.to_csv(csv_path, index=False)

    print(f"📄 Detailed results saved to: {results_json_path}")
    print(f"📊 GPS summary saved to: {csv_path}")

    gps_images = sum(1 for r in results_summary if r['gps_coordinates'])
    metadata_preserved_count = sum(1 for r in results_summary if r['metadata_preserved'])
    print(f"📍 GPS data found in {gps_images}/{len(images)} images")
    print(f"📋 Metadata preserved in {metadata_preserved_count}/{len(images)} output images")

    return results_summary

YOLO11l-cls summary (fused): 94 layers, 12,824,837 parameters, 0 gradients, 49.3 GFLOPs


In [ ]:
reults = classify_benthic(
    input_dir='/content/drive/MyDrive/BlueCARES_Seagrass/field_processing/Classification/data/input',
    output_dir='/content/drive/MyDrive/BlueCARES_Seagrass/field_processing/Classification/data/output'
)

Processing: frame_2025-05-14_08-41-08.jpg
Results saved to /content/drive/MyDrive/BlueCARES_Seagrass/field_processing/Classification/data/output/predictions
  ✓ Metadata copied to output image
Processing: frame_2025-05-14_08-41-09.jpg
Results saved to /content/drive/MyDrive/BlueCARES_Seagrass/field_processing/Classification/data/output/predictions
  ✓ Metadata copied to output image
Processing: frame_2025-05-14_08-41-10.jpg
Results saved to /content/drive/MyDrive/BlueCARES_Seagrass/field_processing/Classification/data/output/predictions
  ✓ Metadata copied to output image
Processing: frame_2025-05-14_08-41-11.jpg
Results saved to /content/drive/MyDrive/BlueCARES_Seagrass/field_processing/Classification/data/output/predictions
  ✓ Metadata copied to output image
Processing: frame_2025-05-14_08-41-12.jpg
Results saved to /content/drive/MyDrive/BlueCARES_Seagrass/field_processing/Classification/data/output/predictions
  ✓ Metadata copied to output image
Processing: frame_2025-05-14_08-41-